In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import optuna
from tqdm.auto import tqdm
import warnings, subprocess, os, gc

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Environment Detection ─────────────────────────────────────────────────────
IS_KAGGLE = os.path.exists("/kaggle/input")

def has_local_gpu():
    try:
        result = subprocess.run(["nvidia-smi"], capture_output=True, timeout=5)
        return result.returncode == 0
    except Exception:
        return False

USE_GPU  = IS_KAGGLE or has_local_gpu()
DATA_DIR = "/kaggle/input/playground-series-s6e3/" if IS_KAGGLE else "../data/"
SUB_DIR  = "/kaggle/working/"                       if IS_KAGGLE else "../"

print(f"Environment  : {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"GPU          : {'Enabled ✓' if USE_GPU else 'Disabled — CPU'}")
print(f"Data dir     : {DATA_DIR}")
print(f"Output dir   : {SUB_DIR}")
print(f"XGBoost ver  : {xgb.__version__}")

# ── Settings ─────────────────────────────────────────────────────────────────
RUN_TUNING   = True
N_TRIALS     = 50
N_CV_FOLDS   = 5
SEEDS        = [42, 2024, 777, 1337, 999]
N_SPLITS     = 10
TOTAL_MODELS = len(SEEDS) * N_SPLITS
TARGET       = "Churn"

PRECOMPUTED_PARAMS = {
    "learning_rate":      0.02,
    "max_depth":          6,
    "max_leaves":         50,
    "min_child_weight":   5,
    "subsample":          0.8,
    "colsample_bytree":   0.8,
    "colsample_bylevel":  0.8,
    "reg_alpha":          0.1,
    "reg_lambda":         1.0,
    "gamma":              0.05,
}

In [ ]:
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

print(f"Train : {train.shape}  |  Test : {test.shape}")

# Target
train[TARGET] = (train[TARGET] == "Yes").astype(int)
churn_rate    = train[TARGET].mean()
scale_pos_w   = round((1 - churn_rate) / churn_rate, 4)
print(f"Churn rate       : {churn_rate:.4f}")
print(f"scale_pos_weight : {scale_pos_w}")

# Submission template
sub     = test[["id"]].copy()
test_id = test["id"].copy()

# Joint preprocessing — combine to avoid category mismatch
combined      = pd.concat([train.drop(TARGET, axis=1), test], axis=0).reset_index(drop=True)
train_idx     = range(len(train))
test_idx      = range(len(train), len(train) + len(test))

# Fill nulls
num_cols = combined.select_dtypes(include=np.number).columns.tolist()
cat_cols = combined.select_dtypes(include="object").columns.tolist()
cat_cols = [c for c in cat_cols if c != "id"]

for c in num_cols:
    combined[c].fillna(combined[c].median(), inplace=True)
for c in cat_cols:
    combined[c].fillna("Missing", inplace=True)

print(f"\nNumeric cols  ({len(num_cols)}): {num_cols}")
print(f"Categorical cols ({len(cat_cols)}): {cat_cols}")

In [ ]:
def build_features(df):
    """All feature engineering — existing + new. Works on combined df."""
    d = df.copy()

    # ── EXISTING FE (from previous notebooks) ────────────────────────────────
    num_base = ["tenure", "MonthlyCharges", "TotalCharges"]
    d["num_sum"]  = d[num_base].sum(axis=1)
    d["num_mean"] = d[num_base].mean(axis=1)
    d["num_std"]  = d[num_base].std(axis=1)
    d["num_max"]  = d[num_base].max(axis=1)
    d["num_min"]  = d[num_base].min(axis=1)

    d["Average_Monthly_Actual"] = d["TotalCharges"] / (d["tenure"] + 1e-5)
    d["Monthly_diff"]           = d["MonthlyCharges"] - d["Average_Monthly_Actual"]
    d["Monthly_ratio"]          = d["MonthlyCharges"] / (d["Average_Monthly_Actual"] + 1e-5)

    # ── NEW FE 1: Service Count Features ─────────────────────────────────────
    # Binary yes/no services → count how many a customer has
    service_cols = [
        "PhoneService", "MultipleLines",
        "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport",
        "StreamingTV", "StreamingMovies",
    ]
    for c in service_cols:
        d[f"{c}_bin"] = (d[c] == "Yes").astype(int)

    d["n_services_total"]  = d[[f"{c}_bin" for c in service_cols]].sum(axis=1)
    d["n_streaming"]       = d["StreamingTV_bin"] + d["StreamingMovies_bin"]
    d["n_security"]        = (d["OnlineSecurity_bin"] + d["OnlineBackup_bin"]
                              + d["DeviceProtection_bin"] + d["TechSupport_bin"])
    d["has_internet"]      = (d["InternetService"] != "No").astype(int)
    d["has_phone"]         = (d["PhoneService"] == "Yes").astype(int)

    # ── NEW FE 2: Payment / Billing Features ─────────────────────────────────
    d["is_autopay"]           = d["PaymentMethod"].str.contains("automatic", case=False).astype(int)
    d["is_paperless_autopay"] = ((d["PaperlessBilling"] == "Yes") & (d["is_autopay"] == 1)).astype(int)

    # Expected total vs actual — detects pricing changes / discounts / arrears
    d["expected_total"]    = d["MonthlyCharges"] * d["tenure"]
    d["charge_gap"]        = d["TotalCharges"] - d["expected_total"]
    d["charge_consistency"]= d["TotalCharges"] / (d["expected_total"] + 1e-5)

    # ── NEW FE 3: Tenure Segments ─────────────────────────────────────────────
    d["is_new_customer"]    = (d["tenure"] <= 6).astype(int)
    d["is_mature_customer"] = (d["tenure"] > 48).astype(int)
    d["tenure_bin"]         = pd.cut(
        d["tenure"],
        bins=[0, 6, 12, 24, 48, 72],
        labels=[0, 1, 2, 3, 4]
    ).astype(int)

    # ── NEW FE 4: Log Transforms (tame skewed distributions) ─────────────────
    d["log_tenure"]        = np.log1p(d["tenure"])
    d["log_TotalCharges"]  = np.log1p(d["TotalCharges"])
    d["log_MonthlyCharges"]= np.log1p(d["MonthlyCharges"])

    # ── NEW FE 5: Interaction Features ───────────────────────────────────────
    d["tenure_x_monthly"]   = d["tenure"] * d["MonthlyCharges"]
    d["senior_alone"]       = (
        (d["SeniorCitizen"] == 1) &
        (d["Partner"] == "No") &
        (d["Dependents"] == "No")
    ).astype(int)

    # Contract ordinal: month-to-month=0, one year=1, two year=2
    contract_map = {"Month-to-month": 0, "One year": 1, "Two year": 2}
    d["contract_ord"]       = d["Contract"].map(contract_map).fillna(0).astype(int)
    d["tenure_x_contract"]  = d["tenure"] * d["contract_ord"]

    # High-value at-risk: high monthly charge + month-to-month
    d["high_value_risk"]    = (
        (d["MonthlyCharges"] > d["MonthlyCharges"].median()) &
        (d["contract_ord"] == 0)
    ).astype(int)

    # ── NEW FE 6: Ratio Features ──────────────────────────────────────────────
    d["monthly_per_service"]= d["MonthlyCharges"] / (d["n_services_total"] + 1)
    d["total_per_service"]  = d["TotalCharges"]   / (d["n_services_total"] + 1)

    return d

combined_fe = build_features(combined)

new_fe_cols = [
    "n_services_total", "n_streaming", "n_security", "has_internet", "has_phone",
    "is_autopay", "is_paperless_autopay",
    "expected_total", "charge_gap", "charge_consistency",
    "is_new_customer", "is_mature_customer", "tenure_bin",
    "log_tenure", "log_TotalCharges", "log_MonthlyCharges",
    "tenure_x_monthly", "senior_alone", "contract_ord",
    "tenure_x_contract", "high_value_risk",
    "monthly_per_service", "total_per_service",
]
existing_fe_cols = [
    "num_sum", "num_mean", "num_std", "num_max", "num_min",
    "Average_Monthly_Actual", "Monthly_diff", "Monthly_ratio",
]
print(f"Existing FE cols : {len(existing_fe_cols)}")
print(f"New FE cols      : {len(new_fe_cols)}")
print(f"Total new cols   : {len(existing_fe_cols) + len(new_fe_cols)}")

In [ ]:
# Label encode categorical cols on combined_fe
le = LabelEncoder()
for c in cat_cols:
    combined_fe[c] = le.fit_transform(combined_fe[c].astype(str))

# Feature list — drop id only
FEATURES = [c for c in combined_fe.columns if c not in ["id", TARGET]]

# Split back
train_fe = combined_fe.iloc[train_idx].copy()
test_fe  = combined_fe.iloc[test_idx].copy()

X      = train_fe[FEATURES]
y      = train[TARGET]
X_test = test_fe[FEATURES]

print(f"X shape      : {X.shape}")
print(f"X_test shape : {X_test.shape}")
print(f"Total features: {len(FEATURES)}")
print(f"\nFeature list (last 23 = new FE):")
for i, f in enumerate(FEATURES):
    marker = " ◀ NEW" if f in new_fe_cols else ""
    print(f"  {i+1:3d}. {f}{marker}")

In [ ]:
DEVICE = "cuda" if USE_GPU else "cpu"
TREE_M = "hist"

base_params = {
    "objective":        "binary:logistic",
    "eval_metric":      "auc",
    "tree_method":      TREE_M,
    "device":           DEVICE,
    "grow_policy":      "lossguide",
    "scale_pos_weight": scale_pos_w,
    "seed":             42,
    "verbosity":        0,
    "nthread":          -1,
}
print(f"XGB device: {DEVICE}  |  grow_policy: lossguide  |  scale_pos_weight: {scale_pos_w}")

dtrain_full = xgb.DMatrix(X, label=y)

if RUN_TUNING:
    print(f"\nStarting Optuna: {N_TRIALS} trials × {N_CV_FOLDS}-fold CV")
    pbar = tqdm(total=N_TRIALS, desc="Optuna Trials", unit="trial")
    best_so_far = [0.0]

    def objective(trial):
        params = {
            **base_params,
            "learning_rate":     trial.suggest_float("learning_rate",     0.005, 0.1,   log=True),
            "max_depth":         trial.suggest_int(  "max_depth",         3,     9),
            "max_leaves":        trial.suggest_int(  "max_leaves",        15,    127),
            "min_child_weight":  trial.suggest_float("min_child_weight",  1,     20),
            "subsample":         trial.suggest_float("subsample",         0.5,   1.0),
            "colsample_bytree":  trial.suggest_float("colsample_bytree",  0.4,   1.0),
            "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.4,   1.0),
            "reg_alpha":         trial.suggest_float("reg_alpha",         1e-8,  10.0,  log=True),
            "reg_lambda":        trial.suggest_float("reg_lambda",        1e-8,  10.0,  log=True),
            "gamma":             trial.suggest_float("gamma",             1e-8,  5.0,   log=True),
        }
        cv = xgb.cv(
            params, dtrain_full,
            num_boost_round = 2000,
            nfold           = N_CV_FOLDS,
            stratified      = True,
            early_stopping_rounds = 50,
            seed            = 42,
            verbose_eval    = False,
        )
        score     = cv["test-auc-mean"].max()
        best_iter = int(cv["test-auc-mean"].idxmax())
        trial.set_user_attr("best_iteration", best_iter)

        if score > best_so_far[0]:
            best_so_far[0] = score
        pbar.set_postfix({"AUC": f"{score:.5f}", "Best": f"{best_so_far[0]:.5f}"})
        pbar.update(1)
        return score

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_TRIALS)
    pbar.close()

    best_trial = study.best_trial
    best_iter  = best_trial.user_attrs.get("best_iteration", 500)
    best_params = {**base_params, **best_trial.params}

    print(f"\n{'='*55}")
    print(f"Best CV AUC    : {best_trial.value:.5f}")
    print(f"Best iteration : {best_iter}")
    for k, v in best_trial.params.items():
        print(f"  {k:25s}: {v}")
    print(f"{'='*55}")

    study.trials_dataframe().to_csv(
        os.path.join(SUB_DIR, "xgb_fe_tuning_results.csv"), index=False
    )

else:
    print("Using PRECOMPUTED_PARAMS.")
    best_params = {**base_params, **PRECOMPUTED_PARAMS}
    best_iter   = 500

In [ ]:
print(f"Multi-Seed Ensemble: {len(SEEDS)} seeds × {N_SPLITS} folds = {TOTAL_MODELS} models\n")

test_preds_sum = np.zeros(len(X_test))
global_oof_sum = np.zeros(len(X))
fold_auc_log   = []

dtest = xgb.DMatrix(X_test)

outer_bar = tqdm(SEEDS, desc="Seeds", position=0)

for seed in outer_bar:
    outer_bar.set_description(f"Seed {seed}")
    skf      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    seed_oof = np.zeros(len(X))

    inner_bar = tqdm(
        enumerate(skf.split(X, y)),
        total    = N_SPLITS,
        desc     = "  Folds",
        position = 1,
        leave    = False,
    )

    for fold, (train_idx, val_idx) in inner_bar:
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval   = xgb.DMatrix(X_val, label=y_val)

        fold_params = {**best_params, "seed": seed}
        model = xgb.train(
            fold_params,
            dtrain,
            num_boost_round  = best_iter + 200,
            evals            = [(dval, "val")],
            early_stopping_rounds = 100,
            verbose_eval     = False,
        )

        val_preds  = model.predict(dval,  iteration_range=(0, model.best_iteration))
        test_preds = model.predict(dtest, iteration_range=(0, model.best_iteration))

        fold_auc = roc_auc_score(y_val, val_preds)
        fold_auc_log.append(fold_auc)

        seed_oof[val_idx]        = val_preds
        global_oof_sum[val_idx] += val_preds
        test_preds_sum          += test_preds

        inner_bar.set_postfix({"fold_auc": f"{fold_auc:.5f}", "best_iter": model.best_iteration})

        del model, dtrain, dval
        gc.collect()

    seed_auc = roc_auc_score(y, seed_oof)
    outer_bar.set_postfix({"seed_oof_auc": f"{seed_auc:.5f}"})

global_oof = global_oof_sum / len(SEEDS)
final_auc  = roc_auc_score(y, global_oof)

print(f"\n{'='*55}")
print(f"Fold AUC — mean : {np.mean(fold_auc_log):.5f}  std: {np.std(fold_auc_log):.5f}")
print(f"Global OOF AUC  : {final_auc:.5f}")
print(f"{'='*55}")

In [ ]:
final_preds = test_preds_sum / TOTAL_MODELS
sub[TARGET] = final_preds

out_file = os.path.join(SUB_DIR, "submission_xgb_rich_fe.csv")
sub.to_csv(out_file, index=False)

print(f"Saved → {out_file}")
print(f"Pred range : [{final_preds.min():.4f}, {final_preds.max():.4f}]")
print(f"Pred mean  : {final_preds.mean():.4f}")
display(sub.head())

In [ ]:
import subprocess

result = subprocess.run([
    "kaggle", "competitions", "submit",
    "-c", "playground-series-s6e3",
    "-f", os.path.join(SUB_DIR, "submission_xgb_rich_fe.csv"),
    "-m", "XGB scaledpos+lossguide rich FE (23 new features) 50-model ensemble",
], capture_output=True, text=True)

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)